In [1]:
import pandas as pd
import json

In [2]:
# 총 100개의 test 표본에서 NER 생성 결과를 불러옴
results = pd.read_csv('/workspace/datas/generated/flan-t5-base_fewnerd-tuned_fewnerd_small.csv')

results = results[:100]

results.head()

,Sentence,NER,generated_ner
0,In the early 1930s the band moved to the Grill...,"{""building-other"": [""Grill Room""], ""building-h...","{ ""building-hotel"": [""Grill Room of the Taft H..."
1,The final season of minor league play Elkin Me...,"{""location-park"": [""Elkin Memorial Park""]}","{ ""location-park"": [""Elkin Memorial Park""]}"
2,"They finished the season 14–19 , 9–9 in C-USA ...","{""event-sportsevent"": [""C-USA play""]}","{ ""organization-sportsleague"": [""C-USA""]}"
3,"The B-52 pilot , Major Larry G. Messinger , la...","{""product-other"": [""B-52""], ""person-other"": [""...","{ ""product-airplane"": [""B-52""], ""person-soldie..."
4,The Austro-Hungarian Navy built and operated t...,"{""organization-other"": [""Austro-Hungarian Navy""]}","{ ""organization-other"": [""Austro-Hungarian Nav..."


In [4]:
count_dict = dict()

for ner_json in results['NER']:
    json_dict = json.loads(ner_json)
    for entity, values in json_dict.items():
        entity = entity.split('-')[0]
        if entity not in count_dict:
            count_dict[entity] = len(values)
        else:
            count_dict[entity] += len(values)

print(count_dict)

{'building': 7, 'location': 68, 'organization': 62, 'event': 11, 'product': 19, 'person': 48, 'other': 19, 'art': 12}


# flan-t5-large 결과 분석

일단 현재 overfitting하는 경향이 나타났고, weight decay 수치를 높여봐도 오히려 못하는 상황

In [2]:
results = pd.read_csv('/workspace/datas/generated/flan-t5-large-ner-json-conll2003_random-fp32-w1e3-lr4e5-full-promp1-tuned-conll2003.csv')

results.head()

,Sentence,NER,types,generated_ner
0,"SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRI...","{""LOC"": [""JAPAN""], ""PER"": [""CHINA""]}",conll2003,"""LOC"": [""JAPAN"", ""CHINA""]"
1,Nadim Ladki,"{""PER"": [""Nadim Ladki""]}",conll2003,"""PER"": [""Nadim Ladki""]"
2,"AL-AIN , United Arab Emirates 1996-12-06","{""LOC"": [""AL-AIN"", ""United Arab Emirates""]}",conll2003,"""LOC"": [""AL-AIN"", ""United Arab Emirates""]"
3,Japan began the defence of their Asian Cup tit...,"{""LOC"": [""Japan"", ""Syria""], ""MISC"": [""Asian Cu...",conll2003,"""LOC"": [""Japan"", ""Syria""], ""MISC"": [""Asian Cup""]"
4,But China saw their luck desert them in the se...,"{""LOC"": [""China"", ""Uzbekistan""]}",conll2003,"""LOC"": [""China"", ""Uzbekistan""]"


In [3]:
def parse_json(json_str):
    try:
        # json 문자열 전처리
        if pd.isna(json_str):
            json_str = ""
        if len(json_str) == 0 or json_str[0] != '{':
            json_str = '{' + json_str + '}'
        
        json_obj = json.loads(json_str)
        return json_obj
    except (json.JSONDecodeError, TypeError):
        return None

In [ ]:
wrong_indices = []
for index, row in results.iterrows():
    ref_json = parse_json(row["NER"])
    pred_json = parse_json(row["generated_ner"])
    
    if str(ref_json) != str(pred_json):
        wrong_indices.append(index)

        print(f"Index {index}: Wrong")

Index 0: Match
Index 12: Match
Index 15: Match
Index 20: Match
Index 25: Match
Index 29: Match
Index 34: Match
Index 39: Match
Index 41: Match
Index 48: Match
Index 49: Match
Index 51: Match
Index 87: Match
Index 145: Match
Index 149: Match
Index 151: Match
Index 152: Match
Index 153: Match
Index 155: Match
Index 157: Match
Index 159: Match
Index 163: Match
Index 165: Match
Index 185: Match
Index 189: Match
Index 199: Match
Index 207: Match
Index 210: Match
Index 212: Match
Index 217: Match
Index 234: Match
Index 235: Match
Index 237: Match
Index 263: Match
Index 305: Match
Index 308: Match
Index 312: Match
Index 314: Match
Index 315: Match
Index 317: Match
Index 334: Match
Index 336: Match
Index 339: Match
Index 341: Match
Index 369: Match
Index 371: Match
Index 374: Match
Index 376: Match
Index 378: Match
Index 379: Match
Index 381: Match
Index 382: Match
Index 390: Match
Index 399: Match
Index 408: Match
Index 426: Match
Index 429: Match
Index 431: Match
Index 432: Match
Index 439: 

In [7]:
wrong_df = results.loc[wrong_indices]

In [9]:
print(len(wrong_df))

543


In [8]:
wrong_df.sample(10)

,Sentence,NER,types,generated_ner
3525,NBA BASKETBALL - FRIDAY 'S RESULTS .,"{""ORG"": [""NBA""]}",conll2003,"""MISC"": [""NBA""]"
955,"Czech ambassador to the United Nations , Karel...","{""LOC"": [""Czech""], ""ORG"": [""United Nations"", ""...",conll2003,"""MISC"": [""Czech"", ""Czechs""], ""PER"": [""Karel Ko..."
1088,Mediterranean oil products were steady to most...,"{""ORG"": [""Elf""]}",conll2003,"""MISC"": [""Mediterranean""], ""ORG"": [""Elf""]"
1897,"Toledo 61,514 0","{""LOC"": [""Toledo""]}",conll2003,"""ORG"": [""Toledo""]"
153,"FIFA 's players ' status committee , meeting i...","{""ORG"": [""FIFA"", ""Udinese""], ""LOC"": [""Barcelon...",conll2003,"""ORG"": [""FIFA""], ""LOC"": [""Barcelona""], ""MISC"":..."
49,"Syria : 24 - Salem Bitar , 3 - Bachar Srour ; ...","{""LOC"": [""Syria""], ""PER"": [""Salem Bitar"", ""Bac...",conll2003,"""LOC"": [""Syria""], ""PER"": [""Salem Bitar"", ""Bach..."
3272,Postponed : Airdrieonians v Clydebank ( to Wed...,"{""ORG"": [""Airdrieonians"", ""Clydebank"", ""East""]}",conll2003,"""ORG"": [""Airdrieonians"", ""Clydebank""], ""LOC"": ..."
871,Bottom team Reggiana are also without a suspen...,"{""ORG"": [""Reggiana""], ""MISC"": [""German""], ""PER...",conll2003,"""ORG"": [""Reggiana""], ""PER"": [""Dietmar Beiersdo..."
1880,U.S. barge rates were lightly quoted Friday on...,"{""LOC"": [""U.S."", ""St. Louis""]}",conll2003,"""LOC"": [""U.S.""], ""ORG"": [""St. Louis Merchants ..."
1071,Trade and Industry Secretary Ian Lang added th...,"{""ORG"": [""Trade and Industry""], ""PER"": [""Ian L...",conll2003,"""PER"": [""Ian Lang""], ""LOC"": [""Britain"", ""Unite..."


# classification inference 결과 살펴보기

In [1]:
import numpy as np
import os
import pandas as pd

In [24]:
result_dir_path = "/workspace/datas/encoder_result/conll2003/flan-t5-base-encoder-switch-ner-custom-class_weight-drop10-smoothing0-cycle10-lr2e-4-cosine_restart"
dataset_csv_path = "/workspace/datas/conll2003/testb.switch.csv"

dataset_df = pd.read_csv(dataset_csv_path)
test_texts = dataset_df['Sentence'].tolist()

labels = np.load(os.path.join(result_dir_path, "test_labels.npy"))
predictions = np.load(os.path.join(result_dir_path, "test_predictions.npy"))

In [14]:
print(predictions.shape)
print(labels.shape)

(14732, 223, 2)
(14732, 223)


In [15]:
predictions_oh = predictions.argmax(axis=2)
print(predictions_oh[1])

[0 0 0 0 0 0 1 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0]


In [17]:
print(labels[1])

[-100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100
 -100 -100 -100 -100 -100 -100    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0 -100 -100 -100 -100 -100
 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100
 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100
 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100
 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100
 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100
 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100
 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100
 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100
 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100
 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100
 -100 

In [25]:
valid_predictions = []
valid_labels = []
for cur_row in zip(predictions_oh, labels):
    cur_preds, cur_labels = cur_row
    cur_valid_preds = []
    cur_valid_labels = []
    for p, l in zip(cur_preds, cur_labels):
        if l != -100:
            cur_valid_preds.append(str(p))
            cur_valid_labels.append(str(l))
    valid_predictions.append(' '.join(cur_valid_preds))
    valid_labels.append(' '.join(cur_valid_labels))

dataset_df['true_labels'] = valid_labels
dataset_df['predicted_labels'] = valid_predictions

In [19]:
pd.set_option('display.max_colwidth', None)

In [26]:
dataset_df.head()

,Sentence,NER,true_labels,predicted_labels
0,"Determine whether or not the named entity of type // MISC // is present in the following sentence // SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRISE DEFEAT .",-100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 0 0 0 0 0 0 0 0 0 0 0 0,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
1,"Determine whether or not the named entity of type // ORG // is present in the following sentence // SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRISE DEFEAT .",-100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 0 0 0 0 0 0 0 0 0 0 0 0,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
2,"Determine whether or not the named entity of type // PER // is present in the following sentence // SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRISE DEFEAT .",-100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 0 0 0 0 0 0 0 1 0 0 0 0,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
3,"Determine whether or not the named entity of type // LOC // is present in the following sentence // SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRISE DEFEAT .",-100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 0 0 1 0 0 0 0 0 0 0 0 0,0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0,0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0
4,Determine whether or not the named entity of type // MISC // is present in the following sentence // Nadim Ladki,-100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 0 0,0 0 0 0 0 0 0,0 0 0 0 0 0 0


In [27]:
correct_rows = dataset_df[dataset_df['predicted_labels'] == dataset_df['true_labels']]

print(f"Correct predictions: {len(correct_rows)} / {len(dataset_df)}")

Correct predictions: 13690 / 14732


In [29]:
incorrect_rows = dataset_df[dataset_df['predicted_labels'] != dataset_df['true_labels']]

print(f"Incorrect predictions: {len(incorrect_rows)} / {len(dataset_df)}")

Incorrect predictions: 1042 / 14732


In [35]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

In [47]:
for row in incorrect_rows.sample(10).itertuples():
    sentence = row.Sentence.split("//")[-1].strip()
    entity_type = row.Sentence.split("//")[-3].strip()
    
    sentence_ids = np.array(tokenizer.encode(sentence))
    true_labels = np.array([int(x) for x in row.true_labels.split()])
    predicted_labels = np.array([int(x) for x in row.predicted_labels.split()])
    
    if len(sentence_ids) != len(true_labels) or len(sentence_ids) != len(predicted_labels):
        print("Length mismatch, skipping...")
        print(f"index: {row.Index}")
        continue
    
    label_indices = true_labels == 1
    predicted_indices = predicted_labels == 1
    label_tokens = tokenizer.decode(sentence_ids[label_indices])
    predicted_tokens = tokenizer.decode(sentence_ids[predicted_indices])
    
    print(f"Index: {row.Index}")
    print(f"Entity Type to find: {entity_type}")
    print(f"Sentence: {sentence}")
    print(f"True Labels: {label_tokens}")
    print(f"Predicted Labels: {predicted_tokens}")
    print("-----")

Index: 4696
Entity Type to find: LOC
Sentence: The longest wait to load on the West Coast was 13 days .
True Labels: 
Predicted Labels: West Coast
-----
Index: 6382
Entity Type to find: ORG
Sentence: Wenchang has built a berth for 5,000 deadweight-tonne container ships at the port and invested 34 million yuan ( $ 4.1 million ) to dredge the harbour , Xinhua said .
True Labels: Xinhua
Predicted Labels: Wenchang Xinhua
-----
Index: 9375
Entity Type to find: ORG
Sentence: John Lewis UK store sales up 4.5 % in week .
True Labels: John Lewis UK
Predicted Labels: John Lewis
-----
Index: 14286
Entity Type to find: MISC
Sentence: PACIFIC DIVISION
True Labels: 
Predicted Labels: PACIFIC DIVISION
-----
Index: 9752
Entity Type to find: PER
Sentence: Seagramd ace 20/11/96 5,000 Japan
True Labels: 
Predicted Labels: Seagramd
-----
Index: 5167
Entity Type to find: LOC
Sentence: Baril said that apart from the group of 150,000 , U.S. and British reconnaissance plans had tracked two much smaller groups

conll2003에서는 특히 대문자 단어에 경도되는 성질이 있는 것 같다. 일종의 short cut path로 작용하고 있다. 학습할 때 이들을 소문자로 일괄 변경한 후 학습하도록 하면 어떨까 하는 생각도 든다.

아니면 일부만 소문자화시키고 일부는 그대로 두고 하는 것도 방법일지도?

# fenerd inference 메모리 문제

16배치 53% -> 1323521~1323536

* 64배치 66% -> 1639424~1639487
* 128배치 66% -> 1639040~1639167
* 192배치 66% -> 1638912~1639103

대략 1639040~1639103 사이에 문제의 장문이 있을 듯

In [8]:
import pandas as pd
import numpy as np

In [2]:
test_set = pd.read_csv("/workspace/datas/fewnerd/supervised/test.switch.csv")

In [ ]:
len_list = list()
for index, row in test_set.iterrows():
    len_list.append(len(row['Sentence'].split()))

print(max(len_list))

318


In [10]:
print(len(test_set) * 0.65)
print(len(test_set) * 0.67)

1615099.2
1664794.56


In [11]:
start_index = 1615099
end_index = 1664794

test_subset = test_set.iloc[start_index:end_index+1]
sub_set_len_list = list()
for index, row in test_subset.iterrows():
    sub_set_len_list.append(len(row['Sentence'].split()))

print(max(sub_set_len_list))

136


In [9]:
np.argmax(len_list)

np.int64(358908)

# 각 데이터셋별 positive 비율

## conll2003

0.05549

## wnut17

0.01374

# 데이터셋 평균 길이 분석

일단 flan-t5-base가 conll2003보다 fewnerd, jnlpba, mit ner 같은 데이터셋에서 훨씬 잘하는데 그 원인이 무엇일까.

혹시 데이터셋 길이. 제공되는 context 양의 차이에서 비롯된 것이 아닐까?

In [1]:
import pandas as pd
import numpy as np

In [2]:
def print_sentence_statistics(df):
    len_list = list()
    for index, row in df.iterrows():
        len_list.append(len(row['Sentence'].split()))
    print(f"Max sentence length: {max(len_list)}")
    print(f"Min sentence length: {min(len_list)}")
    print(f"Mean sentence length: {np.mean(len_list)}")
    print(f"Standard Deviation: {np.std(len_list)}")

## conll2003

In [29]:
df = pd.read_csv("/workspace/datas/conll2003/train.inerd.csv")

df.head()

,Sentence,NER,types
0,EU rejects German call to boycott British lamb .,ORG <TCS> EU <ES> MISC <TCS> German <ES> MISC ...,conll2003
1,Peter Blackburn,PER <TCS> Peter Blackburn <ES>,conll2003
2,BRUSSELS 1996-08-22,LOC <TCS> BRUSSELS <ES>,conll2003
3,The European Commission said on Thursday it di...,ORG <TCS> European Commission <ES> MISC <TCS> ...,conll2003
4,Germany 's representative to the European Unio...,LOC <TCS> Germany <ES> LOC <TCS> Britain <ES> ...,conll2003


In [32]:
print_sentence_statistics(df)

Max sentence length: 113
Min sentence length: 1
Mean sentence length: 13.650473775523823
Standard Deviation: 11.700278437864984


conll2003은 평균 13개의 어절로 구성된 데이터셋

In [33]:
df = pd.read_csv("/workspace/datas/conll2003/testb.json.csv")

df.head()

,Sentence,NER,types
0,"SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRI...","{""LOC"": [""JAPAN""], ""PER"": [""CHINA""]}",conll2003
1,Nadim Ladki,"{""PER"": [""Nadim Ladki""]}",conll2003
2,"AL-AIN , United Arab Emirates 1996-12-06","{""LOC"": [""AL-AIN"", ""United Arab Emirates""]}",conll2003
3,Japan began the defence of their Asian Cup tit...,"{""LOC"": [""Japan"", ""Syria""], ""MISC"": [""Asian Cu...",conll2003
4,But China saw their luck desert them in the se...,"{""LOC"": [""China"", ""Uzbekistan""]}",conll2003


In [34]:
print_sentence_statistics(df)

Max sentence length: 124
Min sentence length: 1
Mean sentence length: 12.670377409720336
Standard Deviation: 11.582822655175832


## wnut17

In [35]:
df = pd.read_csv("/workspace/datas/wnut17/train.inerd2.csv")

df.head()

,Sentence,NER
0,@paulwalk It 's the view from where I 'm livin...,Empire State Building <TCS> location <ES> ESB ...
1,From Green Newsfeed : AHFA extends deadline fo...,AHFA <TCS> group <ES>
2,Pxleyes Top 50 Photography Contest Pictures of...,Pxleyes <TCS> corporation <ES>
3,today is my last day at the office .,NaN
4,"4Dbling 's place til monday , party party part...",4Dbling <TCS> person <ES>


In [36]:
print_sentence_statistics(df)

Max sentence length: 41
Min sentence length: 1
Mean sentence length: 18.482616381850324
Standard Deviation: 7.473454778798041


In [37]:
test_df = pd.read_csv("/workspace/datas/wnut17/test.json.csv")

test_df.head()

,Sentence,NER
0,& gt ; * The soldier was killed when another a...,"{""location"": [""Sonmarg""]}"
1,& gt ; * Police last week evacuated 80 village...,"{""location"": [""Waltengoo Nar""]}"
2,& gt ; * The army on Thursday recovered the bo...,{}
3,& gt ; * The four civilians killed included tw...,{}
4,The bodies of the soldiers were recovered afte...,"{""group"": [""Avalanche Rescue Teams"", ""ART""]}"


In [38]:
print_sentence_statistics(test_df)

Max sentence length: 105
Min sentence length: 1
Mean sentence length: 18.177156177156178
Standard Deviation: 14.366595773507163


## jnlpba

In [3]:
df = pd.read_csv("/workspace/datas/jnlpba/train.inerd2.csv")

df.head()

,Sentence,NER
0,IL-2 gene expression and NF-kappa B activation...,IL-2 gene <TCS> DNA <ES> NF-kappa B <TCS> PROT...
1,Activation of the CD28 surface receptor provid...,CD28 surface receptor <TCS> PROTEIN <ES> inter...
2,In primary T lymphocytes we show that CD28 lig...,primary T lymphocytes <TCS> CELL_TYPE <ES> CD2...
3,Delineation of the CD28 signaling cascade was ...,CD28 <TCS> PROTEIN <ES> protein tyrosine kinas...
4,Our data suggest that lipoxygenase metabolites...,lipoxygenase metabolites <TCS> PROTEIN <ES> IL...


In [4]:
print_sentence_statistics(df)

Max sentence length: 204
Min sentence length: 2
Mean sentence length: 26.558341421330745
Standard Deviation: 11.942837572178135


In [7]:
test_df = pd.read_csv("/workspace/datas/jnlpba/test.inerd2.csv")

test_df.head()

,Sentence,NER
0,Number of glucocorticoid receptors in lymphocy...,glucocorticoid receptors <TCS> PROTEIN <ES> ly...
1,The study demonstrated a decreased level of gl...,glucocorticoid receptors <TCS> PROTEIN <ES> GR...
2,"In the lymphocytes with a high GR number , dex...",lymphocytes <TCS> CELL_TYPE <ES> GR <TCS> PROT...
3,"On the other hand , a decreased GR number resu...",GR <TCS> PROTEIN <ES>
4,These data showed that the sensitivity of lymp...,lymphocytes <TCS> CELL_TYPE <ES> GR <TCS> PROT...


In [8]:
print_sentence_statistics(test_df)

Max sentence length: 208
Min sentence length: 2
Mean sentence length: 26.203060165975103
Standard Deviation: 12.356877776959692


## mit ner

In [43]:
# mit_restaurant

df = pd.read_csv("/workspace/datas/mit_restaurant/train.inerd2.csv")

df.head()

,Sentence,NER
0,34,NaN
1,5 star resturants in my town,5 star <TCS> Rating <ES> in my town <TCS> Loca...
2,98 hong kong restaurant reasonable prices,hong kong <TCS> Restaurant_Name <ES> reasonabl...
3,a great lunch spot but open till 2 a m passims...,open till 2 a m <TCS> Hours <ES> passims kitch...
4,a place that serves soft serve ice cream,soft serve ice cream <TCS> Dish <ES>


In [44]:
print_sentence_statistics(df)

Max sentence length: 35
Min sentence length: 1
Mean sentence length: 9.201860313315926
Standard Deviation: 3.6842428481710647


In [45]:
test_df = pd.read_csv("/workspace/datas/mit_restaurant/test.json.csv")

test_df.head()

,Sentence,NER,types
0,a four star restaurant with a bar,"{""Rating"": [""four star""], ""Location"": [""with a...",mit_restaurant
1,any asian cuisine around,"{""Cuisine"": [""asian""], ""Location"": [""around""]}",mit_restaurant
2,any bbq places open before 5 nearby,"{""Cuisine"": [""bbq""], ""Hours"": [""open before 5""...",mit_restaurant
3,any dancing establishments with reasonable pri...,"{""Location"": [""dancing establishments""], ""Pric...",mit_restaurant
4,any good cheap german restaurants nearby,"{""Price"": [""cheap""], ""Cuisine"": [""german""], ""L...",mit_restaurant


In [46]:
print_sentence_statistics(test_df)

Max sentence length: 26
Min sentence length: 1
Mean sentence length: 9.372781065088757
Standard Deviation: 3.5438117426559743


어찌 보면 context 정보가 충분히 주어졌나 주어지지 않았나의 차이일지도 모르겠다.

## fewnerd

In [47]:
df = pd.read_csv("/workspace/datas/fewnerd/supervised/train.inerd2.csv")

df.head()

,Sentence,NER
0,Paul International airport .,NaN
1,"It starred Hicks 's wife , Ellaline Terriss an...",Hicks <TCS> person-artist/author <ES> Ellaline...
2,`` Time `` magazine said the film was `` a mul...,Time <TCS> art-writtenart <ES> George Axelrod ...
3,Pakistani scientists and engineers ' working a...,IAEA <TCS> organization-other <ES>
4,"In February 2008 , Church 's Chicken entered t...",Church 's Chicken <TCS> organization-company <...


In [48]:
print_sentence_statistics(df)


Max sentence length: 267
Min sentence length: 1
Mean sentence length: 24.494471301615732
Standard Deviation: 12.738852101656951


In [49]:
test_df = pd.read_csv("/workspace/datas/fewnerd/supervised/test.json.csv")

test_df.head()

,Sentence,NER
0,In the early 1930s the band moved to the Grill...,"{""building-other"": [""Grill Room""], ""building-h..."
1,The final season of minor league play Elkin Me...,"{""location-park"": [""Elkin Memorial Park""]}"
2,"They finished the season 14–19 , 9–9 in C-USA ...","{""event-sportsevent"": [""C-USA play""]}"
3,"The B-52 pilot , Major Larry G. Messinger , la...","{""product-other"": [""B-52""], ""person-other"": [""..."
4,The Austro-Hungarian Navy built and operated t...,"{""organization-other"": [""Austro-Hungarian Navy""]}"


In [50]:
print_sentence_statistics(test_df)

Max sentence length: 299
Min sentence length: 1
Mean sentence length: 24.46658521036974
Standard Deviation: 12.60714804364474


## ontonotes5

In [53]:
df = pd.read_csv("/workspace/datas/ontonotes5/train.inerd2.csv")

df.head()

,Sentence,NER
0,People start their own businesses for many rea...,NaN
1,But a chance to fill out sales - tax records i...,one <TCS> CARDINAL <ES>
2,Red tape is the bugaboo of small business .,NaN
3,"Ironically , the person who wants to run his o...",NaN
4,Yet every business owner has to face the mound...,NaN


In [54]:
print_sentence_statistics(df)

Max sentence length: 210
Min sentence length: 1
Mean sentence length: 18.16370736265937
Standard Deviation: 13.844617010897457


In [55]:
test_df = pd.read_csv("/workspace/datas/ontonotes5/test.inerd2.csv")

test_df.head()

,Sentence,NER
0,The following were among Friday 's offerings a...,Friday <TCS> DATE <ES> U.S. <TCS> GPE <ES> non...
1,Dow Chemical Co. --,Dow Chemical Co. -- <TCS> ORG <ES>
2,$ 150 million of 8.55 % senior notes due Oct. ...,$ 150 million <TCS> MONEY <ES> 8.55 % <TCS> PE...
3,"The issue , which is puttable back to the comp...","Oct. 15 , 1999 <TCS> DATE <ES> 50 <TCS> CARDIN..."
4,Rated single - A - 1 by Moody 's Investors Ser...,1 <TCS> CARDINAL <ES> Moody 's Investors Servi...


In [56]:
print_sentence_statistics(test_df)

Max sentence length: 151
Min sentence length: 1
Mean sentence length: 18.484991527475188
Standard Deviation: 13.701558621560503


## genia

In [9]:
df = pd.read_csv("/workspace/datas/genia/train.inerd2.csv")

df.head()

,Sentence,NER
0,IL-2 gene expression and NF-kappa B activation...,IL-2 gene <TCS> DNA <ES> NF-kappa B <TCS> prot...
1,Activation of the CD28 surface receptor provid...,CD28 <TCS> protein <ES> CD28 surface receptor ...
2,In primary T lymphocytes we show that CD28 lig...,primary T lymphocytes <TCS> cell_type <ES> CD2...
3,Delineation of the CD28 signaling cascade was ...,CD28 <TCS> protein <ES> protein tyrosine kinas...
4,Our data suggest that lipoxygenase metabolites...,lipoxygenase <TCS> protein <ES> lipoxygenase m...


In [10]:
print_sentence_statistics(df)

Max sentence length: 149
Min sentence length: 1
Mean sentence length: 25.425946881448446
Standard Deviation: 11.037443506069597


In [11]:
test_df = pd.read_csv("/workspace/datas/genia/test.inerd2.csv")

test_df.head()

,Sentence,NER
0,There is a single methionine codon-initiated o...,methionine codon-initiated open reading frame ...
1,When the homeodomain from HB24 was compared to...,homeodomain <TCS> DNA <ES> HB24 <TCS> DNA <ES>...
2,The HB24 mRNA was absent or present at low lev...,HB24 mRNA <TCS> RNA <ES> T lymphocytes <TCS> c...
3,Characterization of HB24 expression in lymphoi...,HB24 <TCS> DNA <ES>
4,"Positive hybridization was found in thymus , t...",NaN


In [12]:
print_sentence_statistics(test_df)

Max sentence length: 106
Min sentence length: 3
Mean sentence length: 25.986515641855448
Standard Deviation: 11.285991519572057


# 결과 분석하기

어떤 원인으로 틀린 것인지, 가령 contrastive를 했을 때 어떻게 좋아졌는지 분석하기

## 환경 설정하기

In [1]:
import pandas as pd

file_to_analyze = "/workspace/datas/generated/flan-t5-base-first-ner-inerd2_conll2003-conll2003-fp32-w1e3-lr1e4-seed_42-tuned-conll2003.csv"

first_result = pd.read_csv(file_to_analyze)

In [3]:
first_result.head()

,Sentence,NER,generated_ner
0,"SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRI...",JAPAN <TCS> LOC <ES> CHINA <TCS> PER <ES>,JAPAN <TCS> LOC <ES> CHINA <TCS> LOC <ES>
1,Nadim Ladki,Nadim Ladki <TCS> PER <ES>,Nadim Ladki <TCS> PER <ES>
2,"AL-AIN , United Arab Emirates 1996-12-06",AL-AIN <TCS> LOC <ES> United Arab Emirates <TC...,AL-AIN <TCS> LOC <ES> United Arab Emirates <TC...
3,Japan began the defence of their Asian Cup tit...,Japan <TCS> LOC <ES> Asian Cup <TCS> MISC <ES>...,Japan <TCS> LOC <ES> Asian Cup <TCS> MISC <ES>...
4,But China saw their luck desert them in the se...,China <TCS> LOC <ES> Uzbekistan <TCS> LOC <ES>,China <TCS> LOC <ES> Uzbekistan <TCS> LOC <ES>


In [4]:
def inerd_to_ordered_list(inerd_str, inerd_version=1):
    word_list = []
    type_list = []
    if len(inerd_str.strip()) > 0:
        entity_list = inerd_str.split("<ES>")
        for entity_str in entity_list:
            if len(entity_str.strip()) == 0:
                continue
            try:
                entity_item = entity_str.strip().split("<TCS>")
                
                if inerd_version == 1:
                    type = entity_item[0].strip()
                    entity = entity_item[1].strip()
                elif inerd_version == 2:
                    type = entity_item[1].strip()
                    entity = entity_item[0].strip()
                word_list.append(entity)
                type_list.append(type)
            except Exception:
                raise json.JSONDecodeError("Invalid iNERD format.")
    
    return word_list, type_list

def inerd_to_tag_list(word_list, type_list, sentence):
    sentence_words = sentence.split()
    tag_list = list()
    
    if len(word_list) == 0:
        return tag_list
    
    word_idx = 0
    
    for idx, sentence_word in enumerate(sentence_words):
        cur_start_word = word_list[word_idx].split()[0]
        while cur_start_word not in sentence_words[idx:]:
            word_idx += 1
            if word_idx >= len(word_list):
                break
            cur_start_word = word_list[word_idx].split()[0]
        
        while sentence_word == cur_start_word:
            cur_tag_word_len = len(word_list[word_idx].split())
            sentence_match_span = ' '.join(sentence_words[idx:idx + cur_tag_word_len])
            if sentence_match_span == word_list[word_idx]:
                # 매칭 성공
                tag_list.append((idx, idx + cur_tag_word_len, type_list[word_idx]))
                
                word_idx += 1
                if word_idx >= len(word_list):
                    break
                
                cur_start_word = word_list[word_idx].split()[0]
            else:
                break
        
        if word_idx >= len(word_list):
            break
    
    return tag_list

In [8]:
first_result.loc[18]

Sentence           '
NER              NaN
generated_ner    NaN
Name: 18, dtype: object

In [10]:
wrong_data_indices = []

for index, row in first_result.iterrows():
    ref_inerd = row['NER']
    pred_inerd = row['generated_ner']
    
    if pd.isna(ref_inerd):
        ref_inerd = ""
    if pd.isna(pred_inerd):
        pred_inerd = ""
    
    ref_words, ref_types = inerd_to_ordered_list(ref_inerd, inerd_version=2)
    pred_words, pred_types = inerd_to_ordered_list(pred_inerd, inerd_version=2)
    
    ref_tags = inerd_to_tag_list(ref_words, ref_types, row['Sentence'])
    pred_tags = inerd_to_tag_list(pred_words, pred_types, row['Sentence'])
    
    if ref_tags != pred_tags:
        wrong_data_indices.append(index)

print(f"Number of wrong data: {len(wrong_data_indices)} / {len(first_result)}")

Number of wrong data: 494 / 3683


In [17]:
first_wrong_indices = wrong_data_indices

## contrastive한 후 결과

In [18]:
file_to_analyze = "/workspace/datas/generated/flan-t5-base-second-ner-inerd2_conll2003-conll2003_contrastive-fp32-w1e3-lr1e4-seed_42-tuned-conll2003.csv"

second_result = pd.read_csv(file_to_analyze)

In [19]:
wrong_data_indices = []

for index, row in second_result.iterrows():
    ref_inerd = row['NER']
    pred_inerd = row['generated_ner']
    
    if pd.isna(ref_inerd):
        ref_inerd = ""
    if pd.isna(pred_inerd):
        pred_inerd = ""
    
    ref_words, ref_types = inerd_to_ordered_list(ref_inerd, inerd_version=2)
    pred_words, pred_types = inerd_to_ordered_list(pred_inerd, inerd_version=2)
    
    ref_tags = inerd_to_tag_list(ref_words, ref_types, row['Sentence'])
    pred_tags = inerd_to_tag_list(pred_words, pred_types, row['Sentence'])
    
    if ref_tags != pred_tags:
        wrong_data_indices.append(index)

print(f"Number of wrong data: {len(wrong_data_indices)} / {len(second_result)}")

Number of wrong data: 456 / 3683


In [20]:
second_wrong_indices = wrong_data_indices

## 두 결과 합치기

In [24]:
entire_wrong_indices = set(first_wrong_indices) | set(second_wrong_indices)

entire_wrong_indices = sorted(entire_wrong_indices)

In [25]:
wrong_results = first_result.loc[entire_wrong_indices]

wrong_results["second_generated_ner"] = second_result.loc[entire_wrong_indices, "generated_ner"].values

In [26]:
wrong_results.head()

,Sentence,NER,generated_ner,second_generated_ner
0,"SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRI...",JAPAN <TCS> LOC <ES> CHINA <TCS> PER <ES>,JAPAN <TCS> LOC <ES> CHINA <TCS> LOC <ES>,JAPAN <TCS> LOC <ES> CHINA <TCS> LOC <ES>
3,Japan began the defence of their Asian Cup tit...,Japan <TCS> LOC <ES> Asian Cup <TCS> MISC <ES>...,Japan <TCS> LOC <ES> Asian Cup <TCS> MISC <ES>...,Japan <TCS> LOC <ES> Asian Cup <TCS> MISC <ES>...
12,Defender Hassan Abbas rose to intercept a long...,Hassan Abbas <TCS> PER <ES> Bitar <TCS> PER <ES>,Hassan Abbas <TCS> PER <ES> Bitar <TCS> ORG <ES>,Hassan Abbas <TCS> PER <ES> Bitar <TCS> ORG <ES>
15,Bitar pulled off fine saves whenever they did .,Bitar <TCS> PER <ES>,Bitar <TCS> ORG <ES>,Bitar <TCS> ORG <ES>
29,Cuttitta announced his retirement after the 19...,Cuttitta <TCS> PER <ES> 1995 World Cup <TCS> M...,Cuttitta <TCS> PER <ES> World Cup <TCS> MISC <...,Cuttitta <TCS> PER <ES> World Cup <TCS> MISC <...


In [27]:
len(wrong_results)

547

In [28]:
common_wrong_indices = set(first_wrong_indices) & set(second_wrong_indices)
different_wrong_indices = set(entire_wrong_indices) - common_wrong_indices

In [30]:
difference_wrong_indices = sorted(different_wrong_indices)

different_wrong_results = wrong_results.loc[difference_wrong_indices]

이 부분은 contrastive learning의 도입이 어떤 효과를 불러왔는지 분석하는 부분

In [32]:
different_wrong_results

,Sentence,NER,generated_ner,second_generated_ner
34,"Squad : Javier Pertile , Paolo Vaccari , Marce...",Javier Pertile <TCS> PER <ES> Paolo Vaccari <T...,Javier Pertile <TCS> PER <ES> Paolo Vaccari <T...,Javier Pertile <TCS> PER <ES> Paolo Vaccari <T...
45,"A minute later , Bitar produced a good double ...",Bitar <TCS> PER <ES> Kazuyoshi Miura <TCS> PER...,Bitar <TCS> PER <ES> Kazuyoshi Miura <TCS> PER...,Bitar <TCS> ORG <ES> Kazuyoshi Miura <TCS> PER...
47,Japan started the second half brightly but Bit...,Japan <TCS> LOC <ES> Bitar <TCS> PER <ES> Naok...,Japan <TCS> LOC <ES> Bitar <TCS> PER <ES> Naok...,Japan <TCS> LOC <ES> Bitar <TCS> ORG <ES> Naok...
51,FREESTYLE SKIING-WORLD CUP MOGUL RESULTS .,SKIING-WORLD CUP <TCS> MISC <ES>,MOGUL <TCS> LOC <ES>,SKIING-WORLD CUP <TCS> MISC <ES>
119,Astle 9-0-53-1 ( w-1 nb-1 ),Astle <TCS> PER <ES>,Astle <TCS> ORG <ES>,Astle <TCS> PER <ES>
...,...,...,...,...
3472,Standings of National,National <TCS> ORG <ES>,National <TCS> MISC <ES>,National <TCS> ORG <ES>
3523,CHARLOTTE AT SEATTLE,CHARLOTTE <TCS> ORG <ES> SEATTLE <TCS> LOC <ES>,CHARLOTTE <TCS> PER <ES> SEATTLE <TCS> LOC <ES>,CHARLOTTE <TCS> ORG <ES> SEATTLE <TCS> LOC <ES>
3525,NBA BASKETBALL - FRIDAY 'S RESULTS .,NBA <TCS> ORG <ES>,NBA <TCS> MISC <ES>,NBA <TCS> ORG <ES>
3628,With just one league match scheduled before th...,Real <TCS> ORG <ES>,Real <TCS> ORG <ES>,Real <TCS> ORG <ES> Christmas <TCS> MISC <ES>


In [33]:
pd.options.display.max_colwidth = None

In [35]:
import random
random_indices = random.sample(different_wrong_results.index.tolist(), 50)

In [86]:
different_wrong_results.loc[random_indices[49]]

Sentence                It looked like turning into a rout as Hwang Sun Hong rapidly added two more in the seventh and 15th minutes but although the Koreans continued to dominate they failed to add to the score before the interval .
NER                                                                                                                                                                                Hwang Sun Hong <TCS> PER <ES> Koreans <TCS> MISC <ES>
generated_ner                                                                                                                                                                      Hwang Sun Hong <TCS> PER <ES> Koreans <TCS> MISC <ES>
second_generated_ner                                                                                                                                                Hwang <TCS> PER <ES> Sun Hong <TCS> PER <ES> Koreans <TCS> MISC <ES>
Name: 3408, dtype: object

In [87]:
common_wrong_results = wrong_results.loc[sorted(common_wrong_indices)]

random_indices = random.sample(common_wrong_results.index.tolist(), 50)

In [137]:
common_wrong_results.loc[random_indices[49]]

Sentence                   Skinheads attack Bratislava Rabbi - police .
NER                                           Bratislava <TCS> LOC <ES>
generated_ner                           Bratislava Rabbi <TCS> PER <ES>
second_generated_ner    Bratislava <TCS> LOC <ES> Rabbi <TCS> MISC <ES>
Name: 1361, dtype: object

## flan-t5-xl 결과 분석

In [141]:
import pandas as pd

file_to_analyze = "/workspace/datas/generated/flan-t5-xl-first-ner-inerd2_conll2003-conll2003-fp32-w1e2-lr1e4-seed_42-tuned-conll2003.csv"

first_result = pd.read_csv(file_to_analyze)

In [139]:
def inerd_to_ordered_list(inerd_str, inerd_version=1):
    word_list = []
    type_list = []
    if len(inerd_str.strip()) > 0:
        entity_list = inerd_str.split("<ES>")
        for entity_str in entity_list:
            if len(entity_str.strip()) == 0:
                continue
            try:
                entity_item = entity_str.strip().split("<TCS>")
                
                if inerd_version == 1:
                    type = entity_item[0].strip()
                    entity = entity_item[1].strip()
                elif inerd_version == 2:
                    type = entity_item[1].strip()
                    entity = entity_item[0].strip()
                word_list.append(entity)
                type_list.append(type)
            except Exception:
                raise json.JSONDecodeError("Invalid iNERD format.")
    
    return word_list, type_list

def inerd_to_tag_list(word_list, type_list, sentence):
    sentence_words = sentence.split()
    tag_list = list()
    
    if len(word_list) == 0:
        return tag_list
    
    word_idx = 0
    
    for idx, sentence_word in enumerate(sentence_words):
        cur_start_word = word_list[word_idx].split()[0]
        while cur_start_word not in sentence_words[idx:]:
            word_idx += 1
            if word_idx >= len(word_list):
                break
            cur_start_word = word_list[word_idx].split()[0]
        
        while sentence_word == cur_start_word:
            cur_tag_word_len = len(word_list[word_idx].split())
            sentence_match_span = ' '.join(sentence_words[idx:idx + cur_tag_word_len])
            if sentence_match_span == word_list[word_idx]:
                # 매칭 성공
                tag_list.append((idx, idx + cur_tag_word_len, type_list[word_idx]))
                
                word_idx += 1
                if word_idx >= len(word_list):
                    break
                
                cur_start_word = word_list[word_idx].split()[0]
            else:
                break
        
        if word_idx >= len(word_list):
            break
    
    return tag_list

In [142]:
wrong_data_indices = []

for index, row in first_result.iterrows():
    ref_inerd = row['NER']
    pred_inerd = row['generated_ner']
    
    if pd.isna(ref_inerd):
        ref_inerd = ""
    if pd.isna(pred_inerd):
        pred_inerd = ""
    
    ref_words, ref_types = inerd_to_ordered_list(ref_inerd, inerd_version=2)
    pred_words, pred_types = inerd_to_ordered_list(pred_inerd, inerd_version=2)
    
    ref_tags = inerd_to_tag_list(ref_words, ref_types, row['Sentence'])
    pred_tags = inerd_to_tag_list(pred_words, pred_types, row['Sentence'])
    
    if ref_tags != pred_tags:
        wrong_data_indices.append(index)

print(f"Number of wrong data: {len(wrong_data_indices)} / {len(first_result)}")

Number of wrong data: 409 / 3683


In [143]:
random_indices = random.sample(wrong_data_indices, 50)

In [193]:
first_result.loc[random_indices[49]]

Sentence         Luxembourg 's traditional Christmas market , which starts on Saturday and runs to December 24 , has taken to the world wide web as a way of publicising its activities .
NER                                                                                                                                                             Luxembourg <TCS> LOC <ES>
generated_ner                                                                                                                         Luxembourg <TCS> LOC <ES> Christmas <TCS> MISC <ES>
Name: 2036, dtype: object